# 03 · Tune on dev

Settings are chosen on the **dev** split only (50 queries), never on test:
- **Input fields and truncation.** One shared text config for all methods, so every method sees the same text.
- **Stage-1 k** for the two-stage method.

The chosen settings are written to `v2/config.json`, which `04_test_runs` reads.

LLM cells need Jan's Local API Server running.

In [ ]:
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore", message="IProgress not found")

RESEARCH = Path.cwd().parent if Path.cwd().name == "v2" else Path.cwd()
sys.path.insert(0, str(RESEARCH))

import pandas as pd
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)

from src.figures import save, setup_style
setup_style()

In [ ]:
BENCHES = ["random", "concept"]
TEXT_CONFIGS = [("t", 600), ("ta", 600), ("tac", 600)]
CHEAP_METHODS = ["bm25", "tfidf", "minilm", "specter2", "qwen", "embeddinggemma", "bge-reranker"]
RUN_LLM = True
STAGE1_KS = [10, 20, 30]

In [ ]:
from src.pipeline_v2 import RunConfig, run_methods

results = []
for bench in BENCHES:
    for fields, max_chars in TEXT_CONFIGS:
        cfg = RunConfig(bench=bench, fields=fields, max_chars=max_chars, split="dev")
        results.append(run_methods(CHEAP_METHODS, cfg))
cheap = pd.concat(results)
cheap.pivot_table(index=["bench", "method"], columns="text_tag", values="rprec")

### Listwise LLM on dev (about 5 minutes per config and bench)

In [ ]:
llm = []
if RUN_LLM:
    for bench in BENCHES:
        for fields, max_chars in TEXT_CONFIGS:
            cfg = RunConfig(bench=bench, fields=fields, max_chars=max_chars, split="dev")
            llm.append(run_methods(["llama"], cfg))
    llm = pd.concat(llm)
    display(llm[["bench", "text_tag", "mrr", "rprec", "ok", "partial", "failed", "sec_per_query"]])

### Choose the text config

Pick the config with the best mean rank across all methods on dev, both benchmarks together. Using one config for every method keeps the comparison fair.

In [ ]:
dev = pd.concat([cheap] + ([llm] if RUN_LLM else []))
ranks = dev.groupby(["bench", "method"])["rprec"].rank(ascending=False)
mean_rank = ranks.groupby(dev["text_tag"]).mean().sort_values()
display(mean_rank.rename("mean rank (lower is better)").to_frame())
best_tag = mean_rank.index[0]
best_fields, best_max, _ = best_tag.split("_")
print("chosen text config:", best_tag)

In [ ]:
from src.figures import fig_dev_tuning
save(fig_dev_tuning(dev), "03_dev_tuning")

**Figure (03_dev_tuning).** Dev-split R-precision for each method under each text config (`t` = title, `a` = abstract, `c` = concepts; number = characters kept per candidate). Test queries are never used for this choice.

### Stage-1 k for two-stage (Qwen3 → Llama), with the chosen text config

In [ ]:
two_stage = []
if RUN_LLM:
    for bench in BENCHES:
        for k in STAGE1_KS:
            cfg = RunConfig(bench=bench, fields=best_fields, max_chars=int(best_max), split="dev", stage1_k=k)
            two_stage.append(run_methods(["qwen→llama"], cfg).assign(k=k))
    two_stage = pd.concat(two_stage)
    display(two_stage[["bench", "k", "mrr", "rprec", "partial", "sec_per_query"]])
    best_k = int(two_stage.groupby("k")["rprec"].mean().idxmax())
else:
    best_k = 20
print("chosen stage-1 k:", best_k)

In [ ]:
import json
config = {"fields": best_fields, "max_chars": int(best_max), "query_max_chars": 1500, "stage1_k": best_k,
          "chosen_on": "dev", "dev_mean_rank": mean_rank.round(3).to_dict()}
(RESEARCH / "v2" / "config.json").write_text(json.dumps(config, indent=2))
config

### Findings

_Written from the outputs above after the notebook is run._